# 02 Feature Engineering

Create leakage-safe train, validation, and test splits. The preprocessing transformer is fitted only on training data.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.customer_features import TARGET_COLUMN, add_customer_risk_features
from src.risk_pipeline import build_model_pipeline
from src.train_risk_model import split_dataset, save_splits
from src.risk_metrics import split_features_target

df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "customer_risk_snapshot.csv")
train_df, validation_df, test_df = split_dataset(df)
save_splits(train_df, validation_df, test_df)
train_df.shape, validation_df.shape, test_df.shape

In [ ]:
preview = add_customer_risk_features(train_df.drop(columns=[TARGET_COLUMN]).head(5))
preview[["customer_id", "tickets_per_account_month", "spend_per_usage_point", "escalation_rate", "high_severity_ticket_share", "renewal_window_flag"]]

In [ ]:
X_train, y_train = split_features_target(train_df)
X_validation, y_validation = split_features_target(validation_df)

pipeline = build_model_pipeline(LogisticRegression(max_iter=1500, class_weight="balanced"))
pipeline.fit(X_train, y_train)
validation_scores = pipeline.predict_proba(X_validation)[:, 1]
validation_scores[:10]

The fitted `pipeline` contains feature engineering, imputation, scaling, one-hot encoding, and the estimator. Validation data is never used to fit preprocessing.